# FEniCS Demo

## 1D Boundary Value Problem ($\text{C}^0$-$\text{P}^1$ FEM)

We consider the 1D boundary value problem

$$-(p(x)u(x)')' + q(x)u(x) = f(x), \quad x\in (0,1)$$
with Dirichlet boundary conditions
$$u(0)=\alpha,\quad u(1)=\beta$$
This is the same model problem we solved in homework 1, with finite element code we wrote from scratch. Let's look at what impelmenting this in FEniCS could look like.

### Weak Formulation

From the differential form, we get the following weak form. We seek $u \in H^1(0,1)$ such that
$$\int_0^1 p(x)u'(x)v'(x) dx + \int_0^1 q(x)u(x)v(x) dx = \int_0^1 f(x)v(x) dx,\quad \forall v\in H_0^1(0,1)$$
Or
$$a(u,v) = L(v),\quad \forall v \in H_0^1(0,1)$$
where
$$a(u,v) = \int_0^1 p(x)u'v' dx + \int_0^1 q(x)uv dx,\quad L(v) =  \int_0^1 f(x)v dx.$$


In [ ]:
# add import statements to be used in this notebook
import numpy as np
from mpi4py import MPI
from dolfinx import mesh, fem
from dolfinx.fem.petsc import LinearProblem
from dolfinx import plot
import ufl
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
def compute_errors(uh, u_ex):
    e = uh - u_ex
    
    error_L2 = np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(e, e) * ufl.dx)))
    error_H1 = np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(ufl.grad(e), ufl.grad(e)) * ufl.dx)))

    return error_L2, error_H1

In [ ]:
def convergence_study(M, solver):
    # perform a convergence study
    # Ns = [10, 20, 40, 80, 160]
    Ns = 10*2**np.arange(0,M)
    errorsL2 = np.zeros(M)
    errorsH1 = np.zeros(M)

    for k, N in enumerate(Ns):
        domain, V, uh, u_exact, errL2, errH1 = solver(N)
        errorsL2[k] = errL2
        errorsH1[k] = errH1
        orderL2 = 0
        orderH1 = 0
        if k > 0:
            orderL2 = np.log2(errorsL2[k-1]/errorsL2[k])
            orderH1 = np.log2(errorsH1[k-1]/errorsH1[k])
        print(f"k={k}:  N={N:3d}, errL2={errL2:2.4e}, order={orderL2:.2f}, errH1={errH1:2.4e}, order={orderH1:.2f}")

    # plot the errors
    plt.loglog(Ns, np.array(errorsL2), '-o', label='L2 error')
    plt.loglog(Ns, np.array(errorsH1), '-s', color='orange', label='H1 error')
    plt.xlabel("Number of elements (N)")
    plt.ylabel("Error")
    plt.title("Convergence of the 1D FEM Solution")
    plt.grid(True, which="both", ls="--")
    plt.legend()
    plt.show()

In [ ]:
def solve_1d_problem(N):
    # create a uniform mesh
    domain = mesh.create_interval(MPI.COMM_WORLD, N, [0.0, 1.0])
    x = ufl.SpatialCoordinate(domain)

    # define the function space
    V = fem.functionspace(domain, ("Lagrange", 1))

    # define an exact solution for error measurement
    u_exact = x[0]*(x[0] - 1.0)*(ufl.sin(5.0*x[0]) + 3.0*ufl.exp(x[0]))

    # coefficients of the differential equation
    p = x[0] + 1.0
    q = fem.Constant(domain, 0.0)
    f = -ufl.div(p * ufl.grad(u_exact)) + q*u_exact

    # define the Dirichlet boundary condition
    boundary = lambda x: np.isclose(x[0], 0.0) | np.isclose(x[0], 1.0)
    bondary_dofs = fem.locate_dofs_geometrical(V, boundary)  # locate boundary dofs
    uD = fem.Function(V, name="u_exact")
    uD.interpolate(lambda x: x[0]*(x[0] - 1.0)*(np.sin(5.0*x[0]) + 3.0*np.exp(x[0])))
    bc = fem.dirichletbc(uD, bondary_dofs)

    # form the variational problem
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)

    a = (p*ufl.dot(ufl.grad(u), ufl.grad(v)) + q*u*v) * ufl.dx
    L = f*v * ufl.dx

    # solve
    problem = fem.petsc.LinearProblem(a, L, bcs=[bc], petsc_options_prefix="1d_")
    uh = problem.solve()

    # compute errors
    error_L2, error_H1 = compute_errors(uh, u_exact)

    return domain, V, uh, u_exact, error_L2, error_H1

In [ ]:
convergence_study(5, solve_1d_problem)

## 2D Boundary Value Problem ($\text{C}^0$-$\text{P}^1$ FEM)

Let us now consider the following 2D boundary-value problem:

$$-\nabla \cdot (p(\mathbf{x}) \nabla u) + q(\mathbf{x}) u = f(\mathbf{x}), \quad \mathbf{x} \in \Omega$$
$$u(\mathbf{x}) = \alpha(\mathbf{x}), \quad \mathbf{x} \in \partial\Omega$$
where $f=f(\mathbf{x})$ and $\alpha(\mathbf{x})$ are given functions, and the spatial domain is the unit square $\Omega = [0,1] \times [0,1]$, with a boudnary $\partial\Omega$.

### Weak Formulation
We look for $u\in V = \{v\ in H^1(\Omega): v\big|_{\partial \Omega} = \alpha(x,y)\}$ such that
$$a(u,v) = L(v),\quad \forall v\in H_0^1(\Omega)$$
Here,
$$a(u,v) = \int_\Omega p(x,y) \nabla u(x,y) \cdot \nabla v(x,y) + q(x,y) u(x,y) v(x,y) dxdy$$
$$L(v) = \int_\Omega f(x,y) v(x,y) dxdy$$

In [ ]:
def solve_2d_problem(N):
    # create a uniform mesh
    domain =  mesh.create_rectangle(
        MPI.COMM_WORLD,
        [[0.0, 0.0], [1.0, 1.0]],
        [N, N],
        mesh.CellType.triangle
    )
    x = ufl.SpatialCoordinate(domain)

    # define the function space
    V = fem.functionspace(domain, ("Lagrange", 1))

    # define an exact solution for error measurement
    u_exact = x[1]**3 + ufl.sin(5*(x[0] + x[1])) + 2*ufl.exp(x[0])
    # u_exact_grad = ufl.as_vector([ufl.diff(u_exact, x[i]) for i in range(2)])

    # coefficients of the differential equation
    p = fem.Constant(domain, 3.0)
    q = fem.Constant(domain, 2.0)
    f = 152*ufl.sin(5*(x[0]+x[1])) - 2*ufl.exp(x[0]) - 18*x[1] + 2*x[1]**3

    # define the Dirichlet boundary condition
    # determine the degrees of freedom on the boundary of the mesh\n",
    tdim = domain.topology.dim
    fdim = tdim - 1
    domain.topology.create_connectivity(fdim, tdim)
    boundary_facets = mesh.exterior_facet_indices(domain.topology)
    boundary_dofs = fem.locate_dofs_topological(V, fdim, boundary_facets)
    uD = fem.Function(V, name="u_exact")
    uD.interpolate(lambda x: x[1]**3 + np.sin(5*(x[0] + x[1])) + 2*np.exp(x[0]))
    bc = fem.dirichletbc(uD, boundary_dofs)

    # form the variational problem
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)

    a = (p*ufl.dot(ufl.grad(u), ufl.grad(v)) + q*u*v) * ufl.dx
    L = f*v * ufl.dx

    # solve
    problem = fem.petsc.LinearProblem(a, L, bcs=[bc], petsc_options_prefix="2d_")
    uh = problem.solve()

    # compute errors
    error_L2, error_H1 = compute_errors(uh, u_exact)

    return domain, V, uh, u_exact, error_L2, error_H1

In [ ]:
convergence_study(4, solve_2d_problem)

# plot the results for N=20
N = 20
domain, V, uh, u_exact, errL2, errH1 = solve_2d_problem(N)

import pyvista
pyvista.OFF_SCREEN = True
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, tdim)
topology, cell_types, geometry = plot.vtk_mesh(domain, tdim)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
uh_topology, uh_cell_types, uh_geometry = plot.vtk_mesh(V)
uh_grid = pyvista.UnstructuredGrid(uh_topology, uh_cell_types, uh_geometry)
uh_grid.point_data["uh"] = uh.x.array.real
uh_grid.set_active_scalars("uh")
uh_plotter = pyvista.Plotter()
uh_plotter.add_mesh(uh_grid, show_edges=True)
uh_plotter.view_xy()
uh_plotter.show()
